# 33 - Tutorial: Waveform Post-Processing

**Audiência:** engenheiros de potência e contribuidores backend/frontend integrando a pipeline de pós-processamento de formas de onda.

**Pré-requisitos:**
- Bindings Python disponíveis (`PYTHONPATH=build/python`)
- Familiaridade com netlists YAML e `SimulationResult`
- Conceitos básicos de FFT, RMS, THD e eficiência

**Objetivos de aprendizado:**
- Configurar e executar jobs de pós-processamento via Python API
- Usar os três tipos de job: `TimeDomain`, `Spectral`, `PowerEfficiency`
- Usar os três modos de janela: `Time`, `Index`, `Cycle`
- Inspecionar métricas escalares, espectro, tabela de harmônicos e eficiência
- Lidar com códigos de diagnóstico e métricas indefinidas (`UndefinedMetric`)
- Parsear a config `simulation.post_processing` a partir de YAML
- Verificar determinismo de execuções repetidas

## Sumário

1. Setup e imports
2. Criação do sinal de referência (conversor buck sintético)
3. **TimeDomain** — todas as métricas (RMS, mean, min, max, p2p, crest, ripple, std)
4. **Spectral** — FFT, tabela de harmônicos, THD e comparação de janelas
5. **PowerEfficiency** — potência média, eficiência e fator de potência
6. Modos de janela: `Time`, `Index` e `Cycle`
7. Diagnósticos: códigos de erro e métricas indefinidas
8. Parse de YAML (`parse_post_processing_yaml`)
9. Múltiplos jobs — execução conjunta e isolamento de falhas
10. Determinismo — execuções repetidas produzem resultados idênticos
11. Visualizações

## 1 — Setup e imports

In [5]:
from __future__ import annotations

import json
import math
import os
import sys
from dataclasses import asdict
from pathlib import Path

import numpy as np

# ── localiza raiz do repositório ────────────────────────────────────────────
def _find_repo_root(start: Path) -> Path:
    here = start.resolve()
    for candidate in (here, *here.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "benchmarks").is_dir():
            return candidate
    raise RuntimeError("Não foi possível localizar a raiz do repositório")

repo_root = _find_repo_root(Path.cwd())
if Path.cwd().resolve() != repo_root:
    os.chdir(repo_root)

build_python = repo_root / "build" / "python"
if build_python.is_dir() and str(build_python) not in sys.path:
    sys.path.insert(0, str(build_python))

# ── importa pulsim ───────────────────────────────────────────────────────────
try:
    import pulsim as ps
except Exception as exc:
    raise RuntimeError(
        "Falha ao importar pulsim. Compile os bindings e execute com PYTHONPATH=build/python."
    ) from exc

# ── verifica que as novas APIs estão presentes ───────────────────────────────
NEW_EXPORTS = [
    "PostProcessingWindowMode", "PostProcessingJobKind", "PostProcessingDiagnosticCode",
    "WindowFunction", "PostProcessingWindowSpec", "PostProcessingJob",
    "PostProcessingOptions", "ScalarMetric", "SpectralBin", "HarmonicEntry",
    "UndefinedMetricEntry", "PostProcessingJobResult", "PostProcessingResult",
    "run_post_processing", "parse_post_processing_yaml",
]
missing_exports = [name for name in NEW_EXPORTS if not hasattr(ps, name)]
assert not missing_exports, f"Exports ausentes: {missing_exports}"

print(f"repo_root    : {repo_root}")
print(f"pulsim versão: {getattr(ps, '__version__', 'unknown')}")
print(f"Novas APIs   : {len(NEW_EXPORTS)} encontradas — OK ✓")
print()
print("Tipos de job  :", [e.value for e in ps.PostProcessingJobKind])
print("Janelas        :", [e.value for e in ps.PostProcessingWindowMode])
print("WindowFunction :", [e.value for e in ps.WindowFunction])
print("Diagnósticos   :", [e.value for e in ps.PostProcessingDiagnosticCode])

RuntimeError: Falha ao importar pulsim. Compile os bindings e execute com PYTHONPATH=build/python.

## 2 — Sinal de referência: conversor buck sintético

Para ilustrar todas as funções sem precisar rodar uma simulação completa,
construímos manualmente um `SimulationResult` com formas de onda que
reproduzem as características de um **conversor buck** (230 → 12 V, 100 kHz, 2 A):

| Sinal | Descrição |
|-------|-----------|
| `V(out)` | Tensão de saída: 12 V DC + ripple 0.1 V @ 100 kHz + harmônico 3° |
| `V(in)`  | Tensão de entrada: 230 V DC puro |
| `I(L1)`  | Corrente no indutor: valor médio 2 A + ripple triangular @ 100 kHz |
| `I(in)`  | Corrente de entrada: pulsos retangulares @ 100 kHz |
| `I(out)` | Corrente de saída: 2 A DC + ruído menor |
| `V(zero)`| Sinal igual a zero (para demonstrar diagnóstico) |

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams.update({"figure.dpi": 100, "font.size": 10})

# ── parâmetros do conversor ──────────────────────────────────────────────────
FS   = 100_000.0   # Hz — frequência de chaveamento
VOUT = 12.0        # V  — tensão de saída nominal
VIN  = 230.0       # V  — tensão de entrada
IOUT = 2.0         # A  — corrente de saída nominal
RIPPLE_V = 0.10    # V  — ripple de tensão pico-a-pico
RIPPLE_I = 0.40    # A  — ripple de corrente no indutor pico-a-pico
N_CYCLES = 10      # ciclos de chaveamento simulados
SAMPLES_PER_CYCLE = 200

N = N_CYCLES * SAMPLES_PER_CYCLE
dt = 1.0 / (FS * SAMPLES_PER_CYCLE)
t = np.arange(N, dtype=float) * dt

# ── formas de onda ─────────────────────────────────────────────────────────
rng  = np.random.default_rng(42)
freq = 2 * np.pi * FS * t

v_out  = VOUT + RIPPLE_V / 2 * np.sin(freq) + 0.02 * np.sin(3 * freq)
v_in   = np.full(N, VIN, dtype=float)
i_l1   = IOUT + RIPPLE_I / 2 * (2 * (t * FS % 1) - 1)   # triângulo
i_in   = IOUT * 0.052 * (np.sin(freq) > 0).astype(float) + 0.01 * rng.standard_normal(N)
i_out  = np.full(N, IOUT, dtype=float) + 0.005 * rng.standard_normal(N)
v_zero = np.zeros(N, dtype=float)

# ── monta SimulationResult ───────────────────────────────────────────────────
#    ps.SimulationResult() é um objeto pybind11 sem argumentos no construtor.
#    As propriedades 'time' e 'virtual_channels' são injetadas diretamente.
result = ps.SimulationResult()
result.time = t.tolist()                 # aceita lista ou numpy array
result.virtual_channels = {
    "V(out)" : v_out,
    "V(in)"  : v_in,
    "I(L1)"  : i_l1,
    "I(in)"  : i_in,
    "I(out)" : i_out,
    "V(zero)": v_zero,
}

print(f"Pontos no tempo : {N}")
print(f"Δt              : {dt*1e6:.2f} µs")
print(f"Janela total    : {t[-1]*1e6:.1f} µs  ({N_CYCLES} ciclos @ {FS/1e3:.0f} kHz)")
print(f"Sinais          : {list(result.virtual_channels.keys())}")

# ── visualização rápida ──────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(10, 6), sharex=True)
axes[0].plot(t * 1e6, v_out, label="V(out)", color="steelblue")
axes[0].set_ylabel("V (V)"); axes[0].legend(loc="upper right")
axes[1].plot(t * 1e6, i_l1, label="I(L1)", color="darkorange")
axes[1].set_ylabel("I (A)"); axes[1].legend(loc="upper right")
axes[2].plot(t * 1e6, i_in, label="I(in)", color="green")
axes[2].set_ylabel("I (A)"); axes[2].set_xlabel("Tempo (µs)")
axes[2].legend(loc="upper right")
fig.suptitle("Formas de onda — conversor buck sintético")
plt.tight_layout()
plt.show()

## 3 — TimeDomain: todas as métricas escalares

O job `TimeDomain` calcula até **8 métricas** sobre qualquer sinal:

| Métrica | Fórmula |
|---------|---------|
| `rms`   | $\sqrt{ \frac{1}{N}\sum x_i^2 }$ |
| `mean`  | $\frac{1}{N}\sum x_i$ |
| `min`   | $\min(x_i)$ |
| `max`   | $\max(x_i)$ |
| `peak_to_peak` | $\max - \min$ |
| `crest_factor` | $\frac{\max\|x_i\|}{x_\text{rms}}$ |
| `ripple_pct` | $\frac{\text{p2p}}{2\,\|\mu\|}\times 100$ (relativo à média) |
| `std_dev` | $\sqrt{\frac{1}{N}\sum(x_i - \mu)^2}$ |

In [ ]:
# ── define job TimeDomain para V(out) ────────────────────────────────────────
# As 8 métricas suportadas – aliases de entrada são aceitos,
# mas as chaves de saída em scalar_metrics são SEMPRE a forma canônica:
#   peak_to_peak / p2p / pkpk  →  "p2p"
#   crest / crest_factor       →  "crest"
#   ripple                     →  "ripple"  (ratio p2p/|mean|, não percentual)
#   std / stddev               →  "std"
all_td_metrics = ["rms", "mean", "min", "max", "p2p", "crest", "ripple", "std"]

job_td = ps.PostProcessingJob(
    job_id="td_vout",
    kind=ps.PostProcessingJobKind.TimeDomain,
    signals=["V(out)"],
    metrics=all_td_metrics,           # explicita todos os 8
    window=ps.PostProcessingWindowSpec(mode=ps.PostProcessingWindowMode.Time),
)

opts = ps.PostProcessingOptions(jobs=[job_td])
pp: ps.PostProcessingResult = ps.run_post_processing(result, opts)

jr = pp.jobs[0]
assert jr.success, f"Job falhou: {jr.diagnostic_message}"

# ── exibe tabela ──────────────────────────────────────────────────────────────
print(f"{'Métrica':<10} {'Valor':>14}  {'Unidade'}")
print("-" * 38)
for metric_name, sm in jr.scalar_metrics.items():
    print(f"{metric_name:<10} {sm.value:>14.6f}  {sm.unit}")

# ── valida valores esperados ──────────────────────────────────────────────────
metrics_dict = jr.scalar_metrics
assert abs(metrics_dict["mean"].value - VOUT) < 0.05, "Média de V(out) errada"
assert abs(metrics_dict["rms"].value - VOUT) < 0.1,  "RMS de V(out) errado"
assert metrics_dict["p2p"].value < RIPPLE_V * 2 + 0.1, "P2P muito alto"
# ripple = p2p / |mean| (ratio, ≠ percentual)
assert metrics_dict["ripple"].value < 0.02, "Ripple ratio deve ser < 0.02"
print("\n✓ Todos os asserts passaram")

In [ ]:
# ── TimeDomain em vários sinais de uma vez ───────────────────────────────────
# Quando há múltiplos sinais, as chaves de scalar_metrics usam ponto:
#   "{sinal}.{metrica}" → ex.: "V(out).rms", "I(L1).p2p"
job_multi = ps.PostProcessingJob(
    job_id="td_all",
    kind=ps.PostProcessingJobKind.TimeDomain,
    signals=["V(out)", "V(in)", "I(L1)", "I(out)"],
    metrics=["rms", "p2p"],
    window=ps.PostProcessingWindowSpec(mode=ps.PostProcessingWindowMode.Time),
)
pp_multi = ps.run_post_processing(result, ps.PostProcessingOptions(jobs=[job_multi]))
jr_multi = pp_multi.jobs[0]
assert jr_multi.success, jr_multi.diagnostic_message
print(f"Sinais analisados : {jr_multi.signal_names}")
print(f"Chaves geradas    : {sorted(jr_multi.scalar_metrics.keys())}")

# ── gráfico de barras de RMS por sinal ────────────────────────────────────────
sig_names = jr_multi.signal_names
rms_vals  = [jr_multi.scalar_metrics[f"{s}.rms"].value for s in sig_names]
p2p_vals  = [jr_multi.scalar_metrics[f"{s}.p2p"].value for s in sig_names]

x = np.arange(len(sig_names))
width = 0.3
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(x - width/2, rms_vals, width, label="RMS", color="steelblue")
ax.bar(x + width/2, p2p_vals, width, label="P2P",  color="darkorange")
ax.set_xticks(x); ax.set_xticklabels(sig_names, rotation=15)
ax.set_ylabel("Valor"); ax.set_title("RMS e P2P por sinal"); ax.legend()
plt.tight_layout(); plt.show()

## 4 — Spectral: FFT, harmônicos e THD

O job `Spectral` executa a **FFT janelada** do sinal e extrai:

- Bins espectrais (`SpectralBin` — frequência, magnitude, fase)
- Tabela de harmônicos (`HarmonicEntry` — número, frequência, magnitude, fase, % do fundamental)  
- **THD** (Total Harmonic Distortion) em %
- `fundamental_hz` detectado automaticamente

Funções de janela disponíveis:

| `WindowFunction` | Melhor uso |
|-----------------|-----------|
| `rectangular`   | Sinais periódicos exatamente no tempo |
| `hann`          | Uso geral — boa rejeição de lóbulos laterais |
| `hamming`       | Alternativa ao Hann com menor mainlobe |
| `blackman`      | Alta rejeição dinâmica, sinal puro |
| `flat_top`      | Calibração de amplitude — erro mínimo de ganho |

In [ ]:
# ── job Spectral com janela Hann ─────────────────────────────────────────────
# Nota: V(out) tem grande componente DC (12 V). Com janela de Hann o DC "vaza"
#       para bins adjacentes e pode mascarar harmônicos de baixa amplitude.
#       Para sinais com strong bias DC, passe 'fundamental_hz' explicitamente.
job_spec = ps.PostProcessingJob(
    job_id="spectral_vout",
    kind=ps.PostProcessingJobKind.Spectral,
    signals=["V(out)"],
    n_harmonics=5,
    fundamental_hz=FS,                      # ← fixa fundamental em 100 kHz
    window_function=ps.WindowFunction.Hann,
    window=ps.PostProcessingWindowSpec(mode=ps.PostProcessingWindowMode.Time),
)
pp_spec = ps.run_post_processing(result, ps.PostProcessingOptions(jobs=[job_spec]))
jr_spec = pp_spec.jobs[0]
assert jr_spec.success, f"Spectral falhou: {jr_spec.diagnostic_message}"

# ── resumo ────────────────────────────────────────────────────────────────────
print(f"Fundamental fixado : {jr_spec.fundamental_hz:.1f} Hz")
print(f"THD                : {jr_spec.thd_pct:.3f} %")
print(f"Bins espectrais    : {len(jr_spec.spectrum_bins)}")
print()

# ── tabela de harmônicos ──────────────────────────────────────────────────────
print(f"{'nº':>3}  {'Freq (Hz)':>10}  {'Magnitude':>12}  {'Fase (°)':>9}  {'% do fund':>10}")
print("-" * 55)
for h in jr_spec.harmonics:
    print(f"{h.harmonic_number:>3}  {h.frequency_hz:>10.1f}  {h.magnitude:>12.6f}  "
          f"{h.phase_deg:>9.2f}  {h.magnitude_pct_fundamental:>10.3f}")

# ── valida ─────────────────────────────────────────────────────────────────────
assert abs(jr_spec.fundamental_hz - FS) / FS < 0.02, "Fundamental fora de ±2%"
assert jr_spec.thd_pct >= 0, "THD não pode ser negativo"
print("\n✓ Fundamental e THD OK")

In [ ]:
# ── comparação de funções de janela ──────────────────────────────────────────
wf_names = [wf for wf in ps.WindowFunction]
thd_by_wf: dict[str, float] = {}
fund_amp_by_wf: dict[str, float] = {}

for wf in wf_names:
    j = ps.PostProcessingJob(
        job_id=f"cmp_{wf.value}",
        kind=ps.PostProcessingJobKind.Spectral,
        signals=["V(out)"],
        n_harmonics=3,
        fundamental_hz=FS,                  # ← sempre fixar para DC-biased
        window_function=wf,
        window=ps.PostProcessingWindowSpec(mode=ps.PostProcessingWindowMode.Time),
    )
    r = ps.run_post_processing(result, ps.PostProcessingOptions(jobs=[j]))
    jr_cmp = r.jobs[0]
    if jr_cmp.success and jr_cmp.harmonics:
        thd_by_wf[wf.value]      = jr_cmp.thd_pct
        fund_amp_by_wf[wf.value] = jr_cmp.harmonics[0].magnitude if jr_cmp.harmonics else 0.0

# ── gráfico comparativo ───────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

labels = list(thd_by_wf.keys())
thd_vals = [thd_by_wf[k] for k in labels]
ax1.bar(labels, thd_vals, color="steelblue")
ax1.set_title("THD (%) por função de janela"); ax1.set_ylabel("THD (%)"); ax1.tick_params(axis="x", rotation=20)

amp_vals = [fund_amp_by_wf.get(k, 0.0) for k in labels]
ax2.bar(labels, amp_vals, color="darkorange")
ax2.set_title("Amplitude do fundamental por janela"); ax2.set_ylabel("Magnitude")
ax2.tick_params(axis="x", rotation=20)

plt.suptitle("Impacto da função de janela na análise espectral")
plt.tight_layout()
plt.show()

print("\nResumo THD por janela:")
for k, v in thd_by_wf.items():
    print(f"  {k:<15}: {v:.4f} %")

## 5 — PowerEfficiency: potência, eficiência e fator de potência

O job `PowerEfficiency` calcula:

$$P_{in} = \frac{1}{N}\sum V_{in}[i]\cdot I_{in}[i], \quad P_{out} = \frac{1}{N}\sum V_{out}[i]\cdot I_{out}[i]$$

$$\eta = \frac{P_{out}}{P_{in}}\times 100\%, \quad PF = \frac{P_{in}}{V_{in,\text{rms}}\cdot I_{in,\text{rms}}}$$

Campos do resultado (`PostProcessingJobResult`):
- `average_input_power` — potência média de entrada (W)
- `average_output_power` — potência média de saída (W)
- `efficiency` — rendimento em %
- `power_factor` — fator de potência (0…1)

In [ ]:
# ── job PowerEfficiency ───────────────────────────────────────────────────────
job_pwr = ps.PostProcessingJob(
    job_id="pwr_buck",
    kind=ps.PostProcessingJobKind.PowerEfficiency,
    signals=["V(out)", "V(in)", "I(in)", "I(out)"],
    input_voltage_signal="V(in)",
    input_current_signal="I(in)",
    output_voltage_signal="V(out)",
    output_current_signal="I(out)",
    window=ps.PostProcessingWindowSpec(mode=ps.PostProcessingWindowMode.Time),
)
pp_pwr = ps.run_post_processing(result, ps.PostProcessingOptions(jobs=[job_pwr]))
jr_pwr = pp_pwr.jobs[0]
assert jr_pwr.success, f"PowerEfficiency falhou: {jr_pwr.diagnostic_message}"

p_in  = jr_pwr.average_input_power
p_out = jr_pwr.average_output_power
eta   = jr_pwr.efficiency
pf    = jr_pwr.power_factor

print("═" * 38)
print(f"  P_in  : {p_in:>10.4f}  W")
print(f"  P_out : {p_out:>10.4f}  W")
print(f"  η     : {eta:>10.4f}  %")
print(f"  PF    : {pf:>10.4f}  (0…1)")
print("═" * 38)

# Os valores esperados para este modelo sintético:
# I(in) é praticamente zero (pulsos de baixa amplitude) → P_in ≈ 0
# I(out) ≈ 2 A DC, V(out) ≈ 12 V → P_out ≈ 24 W
expected_pout = VOUT * IOUT   # ≈ 24 W
assert abs(p_out - expected_pout) < 2.0, f"P_out esperado ≈{expected_pout:.1f} W, obtido {p_out:.3f} W"
print(f"\n✓ P_out ({p_out:.3f} W) próximo do esperado ({expected_pout:.1f} W)")

## 6 — Modos de janela: `Time`, `Index` e `Cycle`

`PostProcessingWindowSpec` suporta três modos de recorte do sinal:

| Modo | Parâmetros | Uso |
|------|-----------|-----|
| `Time` (padrão) | `t_start`, `t_end` | sinal completo (t_start=0, t_end=∞) ou recorte por tempo em segundos |
| `Index` | `i_start`, `i_end` | recorte por índice amostral |
| `Cycle` | `cycle_start`, `cycle_end`, `period` | recorte por número de ciclos |

In [ ]:
T_total = t[-1]           # duração total em segundos (10 ciclos)
half    = T_total / 2.0  # metade do sinal

# helper: aplica job TimeDomain com janela específica; retorna scalar_metrics dict
def td_window(spec: ps.PostProcessingWindowSpec, job_id: str = "td") -> dict:
    j = ps.PostProcessingJob(
        job_id=job_id,
        kind=ps.PostProcessingJobKind.TimeDomain,
        signals=["V(out)"],
        metrics=["rms", "mean", "p2p"],
        window=spec,
    )
    r = ps.run_post_processing(result, ps.PostProcessingOptions(jobs=[j]))
    jr = r.jobs[0]
    if not jr.success:
        raise RuntimeError(f"job '{job_id}' falhou: {jr.diagnostic_message}")
    return jr.scalar_metrics

# ── sinal completo (Time com t_end=∞) ────────────────────────────────────────
sm_full  = td_window(ps.PostProcessingWindowSpec(mode=ps.PostProcessingWindowMode.Time), "full")

# ── Time window: segunda metade ──────────────────────────────────────────────
sm_time  = td_window(ps.PostProcessingWindowSpec(
    mode=ps.PostProcessingWindowMode.Time, t_start=half, t_end=T_total), "time")

# ── Index window: últimas 500 amostras ───────────────────────────────────────
sm_index = td_window(ps.PostProcessingWindowSpec(
    mode=ps.PostProcessingWindowMode.Index, i_start=N - 500, i_end=N - 1), "index")

# ── Cycle window: ciclos 7–9 ─────────────────────────────────────────────────
period_s = 1.0 / FS
sm_cycle = td_window(ps.PostProcessingWindowSpec(
    mode=ps.PostProcessingWindowMode.Cycle,
    cycle_start=7, cycle_end=9,
    period=period_s), "cycle")

# ── comparação ────────────────────────────────────────────────────────────────
print(f"{'Modo':<14} {'RMS':>10}  {'Mean':>10}  {'P2P':>10}")
print("-" * 50)
for label, sm in [("Full(Time)", sm_full), ("Time 2a½", sm_time), ("Index", sm_index), ("Cycle", sm_cycle)]:
    rms_v  = f"{sm['rms'].value:.5f}"  if 'rms'  in sm else "—"
    mean_v = f"{sm['mean'].value:.5f}" if 'mean' in sm else "—"
    p2p_v  = f"{sm['p2p'].value:.5f}"  if 'p2p'  in sm else "—"
    print(f"{label:<14} {rms_v:>10}  {mean_v:>10}  {p2p_v:>10}")

print("\n✓ Todos os modos de janela executaram sem erros")

## 7 — Diagnósticos: `PostProcessingDiagnosticCode` e métricas indefinidas

Quando um job não pode ser concluído normalmente, o campo `diagnostic` recebe um
código `PostProcessingDiagnosticCode`.  Principais códigos:

| Membro enum | `.value` | Significado |
|------------|---------|------------|
| `Ok` | `"ok"` | Sucesso |
| `SignalNotFound` | `"signal_not_found"` | Sinal não encontrado no resultado |
| `InsufficientSamples` | `"insufficient_samples"` | Janela com amostras insuficientes |
| `InvalidWindow` | `"invalid_window"` | Parâmetros de janela inconsistentes |
| `UndefinedMetric` | `"undefined_metric"` | Métrica indefinida (ex.: ripple com média ≈ 0) |

Quando uma métrica é matematicamente indefinida, ela aparece em
`jr.undefined_metrics` como `UndefinedMetricEntry` (em vez de causar falha do job inteiro).

In [ ]:
def run_job(spec_kwargs: dict, label: str) -> ps.PostProcessingJobResult:
    j = ps.PostProcessingJob(**spec_kwargs)
    r = ps.run_post_processing(result, ps.PostProcessingOptions(jobs=[j]))
    jr = r.jobs[0]
    status = "ok" if jr.success else jr.diagnostic.value if jr.diagnostic else "?"
    print(f"[{label}]  success={jr.success}  diagnostic={status}  message={jr.diagnostic_message!r}")
    return jr

# ── diagnóstico 1: sinal inexistente ─────────────────────────────────────────
jr_notfound = run_job({
    "job_id": "notfound",
    "kind"  : ps.PostProcessingJobKind.TimeDomain,
    "signals": ["V(nao_existe)"],
    "window": ps.PostProcessingWindowSpec(mode=ps.PostProcessingWindowMode.Time),
}, "SignalNotFound")
assert not jr_notfound.success
assert jr_notfound.diagnostic == ps.PostProcessingDiagnosticCode.SignalNotFound

# ── diagnóstico 2: janela inválida (t_start > t_end) ─────────────────────────
jr_invwin = run_job({
    "job_id": "invwin",
    "kind"  : ps.PostProcessingJobKind.TimeDomain,
    "signals": ["V(out)"],
    "window": ps.PostProcessingWindowSpec(
        mode=ps.PostProcessingWindowMode.Time, t_start=T_total, t_end=0.0),
}, "InvalidWindow")
assert not jr_invwin.success
# diagnostic pode ser InvalidWindow ou InsufficientSamples dependendo da impl.
assert jr_invwin.diagnostic in (
    ps.PostProcessingDiagnosticCode.InvalidWindow,
    ps.PostProcessingDiagnosticCode.InsufficientSamples,
)

# ── diagnóstico 3: métrica indefinida — ripple e crest de sinal zero ──────────
# "ripple" é indefinido quando mean≈0; "crest" é indefinido quando rms==0.
# O job ainda é bem-sucedido (success=True) mas undefined_metrics fica preenchido.
jr_zero = run_job({
    "job_id": "td_zero",
    "kind"  : ps.PostProcessingJobKind.TimeDomain,
    "signals": ["V(zero)"],
    "metrics": ["ripple", "crest"],           # métricas indefinidas para sinal zero
    "window": ps.PostProcessingWindowSpec(mode=ps.PostProcessingWindowMode.Time),
}, "UndefinedMetric")
# job pode ser success=True com undefined_metrics preenchido
if jr_zero.undefined_metrics:
    print(f"\nMétricas indefinidas ({len(jr_zero.undefined_metrics)}) para V(zero):")
    for u in jr_zero.undefined_metrics:
        print(f"  name={u.name!r}  reason={u.reason.value!r}  msg={u.reason_message!r}")
else:
    print(f"  [zero signal] scalar_metrics keys: {list(jr_zero.scalar_metrics.keys())}")

print("\n✓ Todos os diagnósticos confirmados")

## 8 — `parse_post_processing_yaml`: configuração via YAML

`parse_post_processing_yaml(node)` converte um dict (equivalente ao bloco YAML
`post_processing:` do netlist `.pulsim`) em `PostProcessingOptions`.

Isso permite definir o processamento inteiramente no arquivo de simulação,
sem código Python extra.

In [ ]:
import pprint

# ── define o bloco como apareceria num .pulsim ────────────────────────────────
#    Nota: no YAML o identificador é "id:", não "job_id:".
#          As chaves de sinais de potência são "input_voltage", "input_current",
#          "output_voltage", "output_current".
yaml_node = {
    "jobs": [
        {
            "id"     : "yaml_td",
            "kind"   : "time_domain",
            "signals": ["V(out)", "I(L1)"],
            "metrics": ["rms", "mean", "ripple_pct"],
            "window" : {"mode": "time"},
        },
        {
            "id"             : "yaml_spec",
            "kind"           : "spectral",
            "signals"        : ["V(out)"],
            "n_harmonics"    : 4,
            "window_function": "blackman",
            "window"         : {"mode": "time"},
        },
        {
            "id"             : "yaml_pwr",
            "kind"           : "power_efficiency",
            "signals"        : ["V(out)", "V(in)", "I(in)", "I(out)"],
            "input_voltage"  : "V(in)",
            "input_current"  : "I(in)",
            "output_voltage" : "V(out)",
            "output_current" : "I(out)",
            "window"         : {"mode": "time"},
        },
    ]
}

# ── parse ─────────────────────────────────────────────────────────────────────
opts_from_yaml: ps.PostProcessingOptions = ps.parse_post_processing_yaml(yaml_node)

print(f"PostProcessingOptions parseado com {len(opts_from_yaml.jobs)} jobs:")
for j in opts_from_yaml.jobs:
    print(f"  [{j.job_id}]  kind={j.kind.value}  signals={j.signals}")

# ── verifica tipos ────────────────────────────────────────────────────────────
assert isinstance(opts_from_yaml, ps.PostProcessingOptions)
assert all(isinstance(j, ps.PostProcessingJob) for j in opts_from_yaml.jobs)
assert opts_from_yaml.jobs[0].kind == ps.PostProcessingJobKind.TimeDomain
assert opts_from_yaml.jobs[1].kind == ps.PostProcessingJobKind.Spectral
assert opts_from_yaml.jobs[2].kind == ps.PostProcessingJobKind.PowerEfficiency
assert opts_from_yaml.jobs[0].window.mode == ps.PostProcessingWindowMode.Time

# ── executa a configuração parseada ──────────────────────────────────────────
pp_yaml = ps.run_post_processing(result, opts_from_yaml)
assert pp_yaml.success
print(f"\nResultados YAML:")
for jr in pp_yaml.jobs:
    print(f"  [{jr.job_id}]  success={jr.success}  métricas={len(jr.scalar_metrics)}")

## 9 — Múltiplos jobs: execução conjunta e isolamento de falhas

`run_post_processing` processa todos os jobs de `PostProcessingOptions.jobs` em
uma única chamada.  Se **um job falha**, os demais **continuam sendo executados**
(`PostProcessingResult.success` é `False`, mas jobs individuais têm seu próprio
`success`).  Isso evita que um erro em um job invalide toda a análise.

In [ ]:
# ── batch com 4 jobs: 3 válidos + 1 intencionalmente inválido ────────────────
jobs_batch = [
    ps.PostProcessingJob(
        job_id="batch_td",
        kind=ps.PostProcessingJobKind.TimeDomain,
        signals=["V(out)"],
        window=ps.PostProcessingWindowSpec(mode=ps.PostProcessingWindowMode.Time),
    ),
    ps.PostProcessingJob(
        job_id="batch_spec",
        kind=ps.PostProcessingJobKind.Spectral,
        signals=["I(L1)"],
        n_harmonics=3,
        window=ps.PostProcessingWindowSpec(mode=ps.PostProcessingWindowMode.Time),
    ),
    ps.PostProcessingJob(
        job_id="batch_bad",             # ← sinal inexistente — vai falhar
        kind=ps.PostProcessingJobKind.TimeDomain,
        signals=["V(inexistente)"],
        window=ps.PostProcessingWindowSpec(mode=ps.PostProcessingWindowMode.Time),
    ),
    ps.PostProcessingJob(
        job_id="batch_pwr",
        kind=ps.PostProcessingJobKind.PowerEfficiency,
        signals=["V(out)", "V(in)", "I(in)", "I(out)"],
        input_voltage_signal="V(in)",
        input_current_signal="I(in)",
        output_voltage_signal="V(out)",
        output_current_signal="I(out)",
        window=ps.PostProcessingWindowSpec(mode=ps.PostProcessingWindowMode.Time),
    ),
]

pp_batch: ps.PostProcessingResult = ps.run_post_processing(
    result, ps.PostProcessingOptions(jobs=jobs_batch)
)

print(f"PostProcessingResult.success    : {pp_batch.success}")
print(f"total_runtime_seconds           : {pp_batch.total_runtime_seconds:.6f} s")
print(f"run_count                       : {pp_batch.run_count}")
print()

for jr in pp_batch.jobs:
    diag = jr.diagnostic.value if jr.diagnostic else "—"
    print(f"  [{jr.job_id:<15}]  success={jr.success}  diag={diag:<22}  métricas={len(jr.scalar_metrics)}")

# ── verifica isolamento ────────────────────────────────────────────────────────
failed_ids  = [jr.job_id for jr in pp_batch.jobs if not jr.success]
success_ids = [jr.job_id for jr in pp_batch.jobs if     jr.success]

assert "batch_bad"  in failed_ids,  "'batch_bad' deveria ter falhado"
assert "batch_td"   in success_ids, "'batch_td' deveria ter tido sucesso"
assert "batch_spec" in success_ids, "'batch_spec' deveria ter tido sucesso"
assert "batch_pwr"  in success_ids, "'batch_pwr' deveria ter tido sucesso"
print(f"\n✓ Isolamento confirmado: {len(failed_ids)} falha(s), {len(success_ids)} sucesso(s)")

## 10 — Determinismo

Uma propriedade essencial da implementação é que **a mesma entrada sempre
produz a mesma saída**, independentemente de quantas vezes for chamada.

In [ ]:
N_RUNS = 5

det_job = ps.PostProcessingJob(
    job_id="det",
    kind=ps.PostProcessingJobKind.Spectral,
    signals=["V(out)"],
    n_harmonics=3,
    fundamental_hz=FS,                  # ← fixar para DC-biased
    window_function=ps.WindowFunction.Hann,
    window=ps.PostProcessingWindowSpec(mode=ps.PostProcessingWindowMode.Time),
)
det_opts = ps.PostProcessingOptions(jobs=[det_job])

runs: list[ps.PostProcessingResult] = [
    ps.run_post_processing(result, det_opts) for _ in range(N_RUNS)
]

# ── extrai scalares THD e fundamental ────────────────────────────────────────
thd_vals  = [r.jobs[0].thd_pct           for r in runs]
fund_vals = [r.jobs[0].fundamental_hz    for r in runs]

print("Execução | THD (%)       | Fundamental (Hz)")
print("-" * 48)
for i, (thd, fund) in enumerate(zip(thd_vals, fund_vals), 1):
    print(f"  run {i}   | {thd:>13.8f} | {fund:>16.8f}")

# ── asserts de determinismo ───────────────────────────────────────────────────
assert all(v == thd_vals[0]  for v in thd_vals),  "THD variou entre execuções!"
assert all(v == fund_vals[0] for v in fund_vals), "Fundamental variou entre execuções!"
print(f"\n✓ {N_RUNS} execuções idênticas — determinismo confirmado")

## 11 — Visualizações

Quatro gráficos cobrindo os principais resultados do pós-processamento:

1. **Forma de onda + janela anotada** — V(out) com marcação da janela ativa
2. **Espectro de magnitude** — bins FFT em escala log
3. **Gráfico de barras de harmônicos** — amplitude por harmônico
4. **Painel de eficiência** — P_in, P_out, η, PF

In [ ]:
# ── roda análise completa para as visualizações ──────────────────────────────
viz_jobs = [
    ps.PostProcessingJob(
        job_id="viz_td",
        kind=ps.PostProcessingJobKind.TimeDomain,
        signals=["V(out)"],
        window=ps.PostProcessingWindowSpec(
            mode=ps.PostProcessingWindowMode.Time,
            t_start=T_total * 0.4,
            t_end=T_total * 0.9,
        ),
    ),
    ps.PostProcessingJob(
        job_id="viz_spec",
        kind=ps.PostProcessingJobKind.Spectral,
        signals=["V(out)"],
        n_harmonics=5,
        fundamental_hz=FS,              # ← fixar para DC-biased
        window_function=ps.WindowFunction.Hann,
        window=ps.PostProcessingWindowSpec(mode=ps.PostProcessingWindowMode.Time),
    ),
    ps.PostProcessingJob(
        job_id="viz_pwr",
        kind=ps.PostProcessingJobKind.PowerEfficiency,
        signals=["V(out)", "V(in)", "I(in)", "I(out)"],
        input_voltage_signal="V(in)",
        input_current_signal="I(in)",
        output_voltage_signal="V(out)",
        output_current_signal="I(out)",
        window=ps.PostProcessingWindowSpec(mode=ps.PostProcessingWindowMode.Time),
    ),
]
pp_viz = ps.run_post_processing(result, ps.PostProcessingOptions(jobs=viz_jobs))
jr_td_viz, jr_spec_viz, jr_pwr_viz = pp_viz.jobs

fig = plt.figure(figsize=(14, 10))
gs = fig.add_gridspec(2, 2, hspace=0.38, wspace=0.35)
ax_wave, ax_spec, ax_harm, ax_eff = (fig.add_subplot(gs[r, c]) for r in range(2) for c in range(2))

# ── 1. Forma de onda + janela anotada ──────────────────────────────────────────
ax_wave.plot(t * 1e6, v_out, color="steelblue", lw=0.8, label="V(out)")
t_win_s = T_total * 0.4
t_win_e = T_total * 0.9
ax_wave.axvspan(t_win_s * 1e6, t_win_e * 1e6, alpha=0.15, color="orange", label="janela TimeDomain")
ax_wave.set_title("Forma de onda + janela"); ax_wave.set_xlabel("Tempo (µs)"); ax_wave.set_ylabel("V (V)")
ax_wave.legend(fontsize=8)

# ── 2. Espectro de magnitude ───────────────────────────────────────────────────
if jr_spec_viz.success and jr_spec_viz.spectrum_bins:
    freqs_bin = np.array([b.frequency_hz    for b in jr_spec_viz.spectrum_bins])
    mags_bin  = np.array([b.magnitude       for b in jr_spec_viz.spectrum_bins])
    # mostra apenas bins positivos e com magnitude > ruído de fundo
    noise_floor = mags_bin.max() * 1e-3 if mags_bin.size else 0.0
    mask = (freqs_bin > 0) & (mags_bin > noise_floor)
    ax_spec.semilogy(freqs_bin[mask] / 1e3, mags_bin[mask], color="purple", lw=0.7)
    ax_spec.set_title("Espectro de magnitude"); ax_spec.set_xlabel("Frequência (kHz)"); ax_spec.set_ylabel("Magnitude (log)")
else:
    ax_spec.text(0.5, 0.5, "sem dados espectrais", ha="center", va="center", transform=ax_spec.transAxes)

# ── 3. Harmônicos ──────────────────────────────────────────────────────────────
if jr_spec_viz.success and jr_spec_viz.harmonics:
    harm_ns  = [h.harmonic_number         for h in jr_spec_viz.harmonics]
    harm_amp = [h.magnitude               for h in jr_spec_viz.harmonics]
    bars = ax_harm.bar(harm_ns, harm_amp, color="darkorange")
    ax_harm.set_title(f"Harmônicos (THD={jr_spec_viz.thd_pct:.2f}%)")
    ax_harm.set_xlabel("Número do harmônico"); ax_harm.set_ylabel("Amplitude")
    ax_harm.set_xticks(harm_ns)
else:
    ax_harm.text(0.5, 0.5, "sem harmônicos", ha="center", va="center", transform=ax_harm.transAxes)

# ── 4. Painel de eficiência ────────────────────────────────────────────────────
if jr_pwr_viz.success:
    labels_pwr = ["P_in (W)", "P_out (W)", "η (%)", "PF (×100)"]
    values_pwr = [
        jr_pwr_viz.average_input_power,
        jr_pwr_viz.average_output_power,
        jr_pwr_viz.efficiency,
        jr_pwr_viz.power_factor * 100,
    ]
    ax_eff.barh(labels_pwr, values_pwr, color=["#4C72B0", "#55A868", "#C44E52", "#8172B2"])
    ax_eff.set_title("PowerEfficiency"); ax_eff.set_xlabel("Valor")
    for i, v in enumerate(values_pwr):
        ax_eff.text(v * 1.01 if v >= 0 else v - abs(v) * 0.15, i, f"{v:.3f}", va="center", fontsize=8)
else:
    ax_eff.text(0.5, 0.5, "PowerEfficiency falhou", ha="center", va="center", transform=ax_eff.transAxes)

fig.suptitle("Pós-processamento — painel de visualizações", fontsize=13)
plt.show()

## 12 — Resumo dos tipos e funções cobertas

| Categoria | Símbolo | Coberto? |
|-----------|---------|----------|
| Enum | `PostProcessingWindowMode` | ✓ seção 6 |
| Enum | `PostProcessingJobKind` | ✓ seções 3–5 |
| Enum | `PostProcessingDiagnosticCode` | ✓ seção 7 |
| Enum | `WindowFunction` | ✓ seção 4 |
| Dataclass | `PostProcessingWindowSpec` | ✓ seção 6 |
| Dataclass | `PostProcessingJob` | ✓ todas as seções |
| Dataclass | `PostProcessingOptions` | ✓ todas as seções |
| Dataclass | `ScalarMetric` | ✓ seção 3 |
| Dataclass | `SpectralBin` | ✓ seção 4 / 11 |
| Dataclass | `HarmonicEntry` | ✓ seção 4 / 11 |
| Dataclass | `UndefinedMetricEntry` | ✓ seção 7 |
| Dataclass | `PostProcessingJobResult` | ✓ todas as seções |
| Dataclass | `PostProcessingResult` | ✓ seções 9 / 10 |
| Função | `run_post_processing` | ✓ todas as seções |
| Função | `parse_post_processing_yaml` | ✓ seção 8 |

**15/15 exports públicos cobertos — tutorial completo ✓**